# Chronological 28-Day Forecasting Experiments

Research: **Evaluating the Effectiveness of AI-Based Demand Forecasting for Retail Businesses Under Volatile Demand Conditions.**

This notebook documents a 5,000-series M5 pilot. It leaves `01_Data_Exploration.ipynb` unchanged. Results are aggregate and reproducible from the committed manifest and report CSVs. The source runner is `src/forecast_experiments.py`.


## Design

Three rolling 28-day chronological folds follow a 365-day training window, with a prior 28-day segment used for early stopping. Sample selection uses seed 42. Volatility is classified at each fold cutoff using terciles of the coefficient of variation of positive-demand weekly totals over the preceding 52 complete weeks; zero-demand-week share is reported separately.

Models: 7-day seasonal naive, four-week weekday mean, recursive LightGBM without price, and LightGBM with scheduled price features. M5 scheduled prices are assumed known throughout the horizon. Price variation is only a proxy for promotions.


In [ ]:
from pathlib import Path
import pandas as pd

# Prefer fresh local run artifacts. Fall back to the committed 5,000-series snapshot.
local = Path("artifacts")
report = Path("reports/m5_5000_pilot")
root = local if (local / "pooled_metrics.csv").exists() else report
pooled = pd.read_csv(root / "pooled_metrics.csv")
fold_name = "metrics.csv" if root == local else "fold_metrics.csv"
folds = pd.read_csv(root / fold_name)
bootstrap = pd.read_csv(root / "bootstrap_comparison.csv")
manifest = pd.read_json(root / "run_manifest.json", typ="series")
print(f"Results loaded from: {root}")
manifest


## Pooled performance

Pooled WAPE = total absolute error divided by total actual demand. Pooled MAE = total absolute error divided by forecast observations. These pool observations across all three folds before scoring.


In [ ]:
all_scores = pooled.loc[pooled["volatility_group"].eq("all")].copy()
all_scores["WAPE"] = all_scores["absolute_error_sum"] / all_scores["actual_demand_sum"]
all_scores["MAE"] = all_scores["absolute_error_sum"] / all_scores["forecast_observations"]
all_scores[["model", "WAPE", "MAE", "folds", "forecast_observations"]].sort_values("WAPE")


## Volatility groups

The table below compares price-aware LightGBM against the four-week weekday-mean baseline. The reported confidence interval resamples series IDs and keeps all fold observations for a selected series together. It is conditional on this seeded 5,000-series sample and is not a p-value.


In [ ]:
group_ci = bootstrap.loc[
    bootstrap["comparison"].eq("baseline")
    & bootstrap["reference_model"].eq("four_week_weekday_mean")
    & bootstrap["candidate_model"].eq("lightgbm_recursive_price"),
    ["volatility_group", "WAPE_improvement", "improvement_95CI_low", "improvement_95CI_high", "bootstrap_replicates"],
].sort_values("volatility_group")
group_ci


## Interpretation and limits

In this pilot, price-aware LightGBM has lower pooled WAPE than both traditional baselines and the no-price LightGBM. Its advantage over the weekday-mean baseline is larger in the moderate- and high-volatility groups. Scheduled price features provide a smaller additional improvement. These findings are preliminary: only 5,000 series were evaluated, the full-population run did not complete within the available compute session, no model tuning was performed, and no independent promotion variable or forecast-uncertainty estimate is included. Run the full population on adequately provisioned compute before making population-level claims.
